Using email to identify users, remove anyone found in both datasets.

In [8]:
import pandas as pd

TRIAL_FILE = "PMH_Scribe_Trial_No_Conversion_List.xlsx"
ACTIVE_FILE = "PMH_subscriptions_(Active)_march.xlsx"
OUTPUT_FILE = "PMH_Scribe_Trial_No_Conversion_CLEANED.xlsx"

TRIAL_SHEET = "Trial No Conversion"
ACTIVE_SHEET = "Sheet 1 - subscriptions (Active"

# Load trial and active data.
trial = pd.read_excel(TRIAL_FILE, sheet_name=TRIAL_SHEET, header=1)
active = pd.read_excel(ACTIVE_FILE, sheet_name=ACTIVE_SHEET)

# Normalize emails.
trial_email = trial["Email"].fillna("").str.strip().str.lower()
active_email = active["Customer Email"].fillna("").str.strip().str.lower()

# Only keep rows NOT in active.
cleaned = trial[~trial_email.isin(active_email)]

# Save rows that were removed.
removed = trial[trial_email.isin(active_email)].copy()

print(f"Rows removed: {len(trial) - len(cleaned)}")
removed_name_email = removed[["Name", "Email"]].sort_values("Email")
removed_name_email

# Normalize email for duplicate check.
cleaned["email_norm"] = cleaned["Email"].fillna("").astype(str).str.strip().str.lower()

# Show duplicated emails.
dupes = cleaned[(cleaned["email_norm"] != "") &(cleaned.duplicated(subset=["email_norm"], keep=False))].copy()

# Print duplicate count and unique count.
print(f"Duplicate rows: {len(dupes)}")
print(f"Unique duplicated emails: {dupes['Email'].str.strip().str.lower().nunique()}")

# Display duplicate rows.
dupes_view = dupes[["Name", "Email"]].sort_values("Email").reset_index(drop=True)
dupes_view

# Save to new xlsx file.
cleaned.to_excel(OUTPUT_FILE, index=False)

Rows removed: 82
Duplicate rows: 58
Unique duplicated emails: 28


In [ ]:
import sqlite3

# Normalize emails, remove active subscribers, and profile duplicates.
conn = sqlite3.connect(":memory:")
trial.to_sql("trial_no_conversion", conn, index=False, if_exists="replace")
active.to_sql("active_subscriptions", conn, index=False, if_exists="replace")

# Keep records whose email is not in active.
cleaned_sql = pd.read_sql_query("""
WITH trial_norm AS (
    SELECT
        t.*, 
        LOWER(TRIM(COALESCE(t.Email, ''))) AS email_norm
    FROM trial_no_conversion t
),
active_norm AS (
    SELECT DISTINCT
        LOWER(TRIM(COALESCE(a.[Customer Email], ''))) AS email_norm
    FROM active_subscriptions a
)
SELECT t.*
FROM trial_norm t
LEFT JOIN active_norm a
    ON t.email_norm = a.email_norm
WHERE a.email_norm IS NULL
""", conn)

# Store removed rows.
removed_sql = pd.read_sql_query("""
WITH trial_norm AS (
    SELECT
        t.*, 
        LOWER(TRIM(COALESCE(t.Email, ''))) AS email_norm
    FROM trial_no_conversion t
),
active_norm AS (
    SELECT DISTINCT
        LOWER(TRIM(COALESCE(a.[Customer Email], ''))) AS email_norm
    FROM active_subscriptions a
)
SELECT t.*
FROM trial_norm t
JOIN active_norm a
    ON t.email_norm = a.email_norm
""", conn)

print(f"Rows removed (SQL): {len(removed_sql)}")
removed_name_email_sql = removed_sql[["Name", "Email"]].sort_values("Email")

# Identify duplicate emails in the cleaned output.
cleaned_sql["email_norm"] = cleaned_sql["Email"].fillna("").astype(str).str.strip().str.lower()
dupes_sql = cleaned_sql[
    (cleaned_sql["email_norm"] != "")
    & (cleaned_sql.duplicated(subset=["email_norm"], keep=False))
].copy()

print(f"Duplicate rows (SQL): {len(dupes_sql)}")
print(f"Unique duplicated emails (SQL): {dupes_sql['email_norm'].nunique()}")

dupes_view_sql = dupes_sql[["Name", "Email"]].sort_values("Email").reset_index(drop=True)

# Save SQL-cleaned output.
cleaned_sql.to_excel("PMH_Scribe_Trial_No_Conversion_CLEANED_SQL.xlsx", index=False)

conn.close()

Rows removed (SQL): 82
Duplicate rows (SQL): 58
Unique duplicated emails (SQL): 28
